# HackUDC 2026 — Inditex Fashion Retrieval (Colab Experiment Notebook)
**Task:** Given a bundle (model photo with multiple garments), retrieve up to 15 matching product IDs from a 27K-product catalog.  
**Metric:** Recall@15

> This notebook is the **experiment branch** — it runs new retrieval strategies in parallel with the Kaggle baseline.  
> Key additions: Reciprocal Rank Fusion, Whole-Image Ensemble, Query Expansion, Score Normalization, Multi-Scale Crops.


## Cell 1: Setup & Google Drive Mount

In [ ]:
from google.colab import drive
import subprocess, sys

# Mount Google Drive
drive.mount('/content/drive', force_remount=False)

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

_pip("fashion-clip", "transformers>=4.35.0", "accelerate", "tqdm", "Pillow", "requests", "open_clip_torch")

try:
    _pip("faiss-gpu")
except Exception:
    try:
        _pip("faiss-cpu")
    except Exception:
        pass

print("✓ Packages installed.")


## Cell 2: Imports & Configuration

In [ ]:
import os
import gc
import json
import math
import time
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter

import numpy as np
import pandas as pd
import requests
import torch
try:
    import faiss
except ModuleNotFoundError:
    raise ModuleNotFoundError("faiss not found. Re-run Cell 1.")
from PIL import Image
from tqdm.auto import tqdm

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

warnings.filterwarnings("ignore")

# ── Paths (Google Drive) ──────────────────────────────────────────────────────
BASE_DIR  = Path("/content/drive/MyDrive/GCED/hackudc")
DATA_DIR  = BASE_DIR / "data"
WORK_DIR  = BASE_DIR / "working"
IMG_DIR   = WORK_DIR / "images"
PROD_DIR  = IMG_DIR / "products"
BUND_DIR  = IMG_DIR / "bundles"
SUBM_FILE = WORK_DIR / "submission.csv"

for d in [PROD_DIR, BUND_DIR, WORK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Base dir: {BASE_DIR}")
print(f"Data dir: {DATA_DIR}")
print(f"Work dir: {WORK_DIR}")

# ── Hardware ──────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Model selection ───────────────────────────────────────────────────────────
USE_MARQO      = True
MARQO_MODEL_ID = "marqo/marqo-fashionSigLIP"
MODEL_TAG      = "marqo" if USE_MARQO else "fclip"
EMB_FILE       = WORK_DIR / f"product_embeddings_{MODEL_TAG}.npy"
IDS_FILE       = WORK_DIR / f"product_ids_{MODEL_TAG}.json"

# ── Core hyper-params ─────────────────────────────────────────────────────────
TOP_K              = 15
EMBED_BATCH        = 32    # Lower for Colab (shared VRAM)
IMG_SIZE           = 224

# ── Download ──────────────────────────────────────────────────────────────────
DOWNLOAD_WORKERS   = 16   # Colab CPU limits
DOWNLOAD_TIMEOUT   = 20
DOWNLOAD_RETRIES   = 3
DOWNLOAD_HEADERS   = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
}

# ── Retrieval ─────────────────────────────────────────────────────────────────
K_PER_SEGMENT      = 150   # candidates per segment before category filter
AGGREGATION        = "sum" # "sum" or "rrf" — toggled in Cell 9

# ── Text re-ranking ───────────────────────────────────────────────────────────
ENABLE_TEXT_RERANK = True
TEXT_RERANK_ALPHA  = 0.20  # 0=pure image, 1=pure text
TEXT_RERANK_TOP_N  = 50

# ── Experiment flags (Part B — colab-only) ────────────────────────────────────
ENABLE_RRF                  = True   # Reciprocal Rank Fusion aggregation
RRF_K                       = 60     # RRF constant
ENSEMBLE_WHOLE_IMAGE        = True   # Blend full-bundle score into final
ENSEMBLE_WHOLE_WEIGHT       = 0.30   # Weight for whole-image score
ENABLE_QUERY_EXPANSION      = True   # Pseudo-relevance feedback
QE_TOP_N                    = 3      # Products to use for query expansion
QE_ALPHA                    = 0.60   # Weight of original query (1-QE_ALPHA for expanded)
NORMALIZE_SEGMENT_SCORES    = True   # Min-max normalize per segment before summing
MULTI_SCALE_CROPS           = True   # Embed crops at multiple padding scales
SCALE_FACTORS               = [1.0, 1.3, 1.7]  # Relative padding multipliers

print("\n✓ Configuration loaded.")


## Cell 3: Data Loading

In [ ]:
print("Loading CSV files from:", DATA_DIR)
assert DATA_DIR.exists(), f"Data dir not found: {DATA_DIR}\nExpected CSVs in this directory."

bundles_df  = pd.read_csv(DATA_DIR / "bundles_dataset.csv")
products_df = pd.read_csv(DATA_DIR / "product_dataset.csv")
train_df    = pd.read_csv(DATA_DIR / "bundles_product_match_train.csv")
test_df     = pd.read_csv(DATA_DIR / "bundles_product_match_test.csv")

print("=== Dataset Statistics ===")
print(f"Bundles total  : {len(bundles_df):,}")
print(f"Products total : {len(products_df):,}")
print(f"Train pairs    : {len(train_df):,}")
print(f"Test bundles   : {len(test_df['bundle_asset_id'].unique()):,}")

# Ground-truth lookup
train_gt: dict[str, set] = (
    train_df.dropna(subset=["product_asset_id"])
    .groupby("bundle_asset_id")["product_asset_id"]
    .apply(set)
    .to_dict()
)

counts = [len(v) for v in train_gt.values()]
print(f"\nTrain bundles with GT  : {len(train_gt):,}")
print(f"Avg products/bundle    : {np.mean(counts):.2f}")
print(f"Max products/bundle    : {max(counts)}")

# URL & description lookups
bundle_url:   dict[str, str] = dict(zip(bundles_df["bundle_asset_id"], bundles_df["bundle_image_url"]))
product_url:  dict[str, str] = dict(zip(products_df["product_asset_id"], products_df["product_image_url"]))
product_desc: dict[str, str] = dict(zip(products_df["product_asset_id"], products_df["product_description"].fillna("")))

if "product_description" in products_df.columns:
    cat_counts = products_df["product_description"].value_counts().head(10)
    print("\nTop-10 product categories:")
    print(cat_counts.to_string())

test_bundle_ids = test_df["bundle_asset_id"].dropna().unique().tolist()
print(f"\nTest bundle IDs: {len(test_bundle_ids)}")


## Cell 4: Image Download + Preview

In [ ]:
def download_image(
    asset_id: str,
    url: str,
    out_dir: Path,
    timeout: int = DOWNLOAD_TIMEOUT,
    retries: int = DOWNLOAD_RETRIES,
) -> bool:
    """Download image with retries + exponential backoff. Returns True on success."""
    out_path = out_dir / f"{asset_id}.jpg"
    if out_path.exists():
        return True
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=timeout, headers=DOWNLOAD_HEADERS)
            r.raise_for_status()
            if len(r.content) < 500:
                raise ValueError(f"Response too small ({len(r.content)} bytes)")
            out_path.write_bytes(r.content)
            return True
        except Exception:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)  # 1s, 2s, 4s backoff
    return False


def batch_download(
    id_url_pairs: list[tuple[str, str]],
    out_dir: Path,
    desc: str = "Downloading",
    timeout: int = DOWNLOAD_TIMEOUT,
) -> tuple[list[str], list[tuple[str, str]]]:
    """Parallel download. Returns (ok_ids, failed_pairs)."""
    ok_ids: list[str] = []
    failed_pairs: list[tuple[str, str]] = []
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        futures = {
            pool.submit(download_image, aid, url, out_dir, timeout): (aid, url)
            for aid, url in id_url_pairs
        }
        for fut in tqdm(as_completed(futures), total=len(futures), desc=desc):
            aid, url = futures[fut]
            if fut.result():
                ok_ids.append(aid)
            else:
                failed_pairs.append((aid, url))
    return ok_ids, failed_pairs


def load_image(asset_id: str, img_dir: Path) -> Image.Image | None:
    p = img_dir / f"{asset_id}.jpg"
    if not p.exists():
        return None
    try:
        return Image.open(p).convert("RGB")
    except Exception:
        return None


# ── Download products (~27K) ──────────────────────────────────────────────────
product_pairs = [(pid, url) for pid, url in product_url.items()]
_prod_ok, _prod_failed = batch_download(product_pairs, PROD_DIR, desc="Products pass-1")
print(f"\nProducts pass-1: {len(_prod_ok):,} ok, {len(_prod_failed):,} failed")

if _prod_failed:
    print(f"Retrying {len(_prod_failed):,} failed products (60s timeout) …")
    _prod_ok2, _prod_failed2 = batch_download(_prod_failed, PROD_DIR, desc="Products pass-2", timeout=60)
    _prod_ok += _prod_ok2
    print(f"  Recovered: {len(_prod_ok2):,}  still failed: {len(_prod_failed2):,}")

valid_product_ids = _prod_ok
print(f"Products downloaded: {len(valid_product_ids):,} / {len(product_pairs):,} ({100*len(valid_product_ids)/len(product_pairs):.1f}%)")

# ── Download bundles (~2.3K) ──────────────────────────────────────────────────
bundle_pairs = [(bid, url) for bid, url in bundle_url.items()]
_bund_ok, _bund_failed = batch_download(bundle_pairs, BUND_DIR, desc="Bundles  pass-1")
print(f"\nBundles  pass-1: {len(_bund_ok):,} ok, {len(_bund_failed):,} failed")

if _bund_failed:
    print(f"Retrying {len(_bund_failed):,} failed bundles (60s timeout) …")
    _bund_ok2, _bund_failed2 = batch_download(_bund_failed, BUND_DIR, desc="Bundles  pass-2", timeout=60)
    _bund_ok += _bund_ok2
    print(f"  Recovered: {len(_bund_ok2):,}  still failed: {len(_bund_failed2):,}")

valid_bundle_ids = _bund_ok
print(f"Bundles downloaded : {len(valid_bundle_ids):,} / {len(bundle_pairs):,} ({100*len(valid_bundle_ids)/len(bundle_pairs):.1f}%)")

valid_product_set = set(valid_product_ids)
valid_bundle_set  = set(valid_bundle_ids)


def show_images(asset_ids, img_dir, title="", n_cols=5):
    imgs = [(aid, load_image(aid, img_dir)) for aid in asset_ids[:n_cols * 3]]
    imgs = [(aid, img) for aid, img in imgs if img is not None]
    if not imgs: return
    n_cols = min(n_cols, len(imgs))
    n_rows = math.ceil(len(imgs) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
    axes = np.array(axes).flatten()
    for ax, (aid, img) in zip(axes, imgs):
        ax.imshow(img); ax.set_title(aid[:12], fontsize=7); ax.axis("off")
    for ax in axes[len(imgs):]: ax.axis("off")
    fig.suptitle(title, fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(WORK_DIR / f"preview_{title.replace(' ','_')}.png", bbox_inches="tight", dpi=80)
    plt.show()

show_images(valid_product_ids[:15], PROD_DIR, title="Product Sample")
show_images(valid_bundle_ids[:5],   BUND_DIR, title="Bundle Sample")


## Cell 5: Embedding Models (FashionCLIP + Marqo-FashionSigLIP)

In [ ]:
from fashion_clip.fashion_clip import FashionCLIP
import fashion_clip.fashion_clip as _fc_module
import open_clip

# ── FashionCLIP compatibility patch ──────────────────────────────────────────
def _patched_load_model(self, name, device=None, auth_token=None):
    from transformers import CLIPModel, CLIPProcessor
    name = _fc_module._MODELS.get(name, name)
    token_kwargs = {"token": auth_token} if auth_token else {}
    model = CLIPModel.from_pretrained(name, **token_kwargs)
    processor = CLIPProcessor.from_pretrained(name, **token_kwargs)
    hash_val = _fc_module._model_processor_hash(name, model, processor)
    return model, processor, hash_val

FashionCLIP._load_model = _patched_load_model

print("Loading FashionCLIP …")
fclip = FashionCLIP("fashion-clip")
if DEVICE == "cuda":
    fclip.model = fclip.model.to(DEVICE)
fclip.model.eval()
print("✓ FashionCLIP loaded.")

def _l2_norm(arr: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(arr, axis=1, keepdims=True).clip(min=1e-8)
    return arr / norms

def _clip_encode_images_raw(imgs):
    inputs = fclip.preprocess(images=imgs, return_tensors="pt", padding=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        out = fclip.model.get_image_features(**{"pixel_values": inputs["pixel_values"]})
    if not isinstance(out, torch.Tensor):
        out = out.image_embeds if hasattr(out, "image_embeds") else out.last_hidden_state[:, 0]
    return out.cpu().float().numpy()

def _clip_encode_texts_raw(texts):
    inputs = fclip.preprocess(text=texts, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    text_inputs = {k: v for k, v in inputs.items() if k in ("input_ids", "attention_mask")}
    with torch.no_grad():
        out = fclip.model.get_text_features(**text_inputs)
    if not isinstance(out, torch.Tensor):
        out = out.text_embeds if hasattr(out, "text_embeds") else out.last_hidden_state[:, 0]
    return out.cpu().float().numpy()

# ── Marqo-FashionSigLIP ───────────────────────────────────────────────────────
if USE_MARQO:
    print(f"Loading Marqo-FashionSigLIP (hf-hub:{MARQO_MODEL_ID}) …")
    marqo_model, _, marqo_preprocess = open_clip.create_model_and_transforms(f"hf-hub:{MARQO_MODEL_ID}")
    marqo_tokenizer = open_clip.get_tokenizer(f"hf-hub:{MARQO_MODEL_ID}")
    marqo_model.to(DEVICE).eval()
    print("✓ Marqo-FashionSigLIP loaded.")

    del fclip
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print("FashionCLIP unloaded to free VRAM.")

def _marqo_encode_images_raw(imgs):
    tensors = torch.stack([marqo_preprocess(img) for img in imgs]).to(DEVICE)
    with torch.no_grad():
        out = marqo_model.encode_image(tensors)
    return out.cpu().float().numpy()

def _marqo_encode_texts_raw(texts):
    tokens = marqo_tokenizer(texts).to(DEVICE)
    with torch.no_grad():
        out = marqo_model.encode_text(tokens)
    return out.cpu().float().numpy()

def _encode_images_raw(imgs):
    return _marqo_encode_images_raw(imgs) if USE_MARQO else _clip_encode_images_raw(imgs)

def _encode_texts_raw(texts):
    return _marqo_encode_texts_raw(texts) if USE_MARQO else _clip_encode_texts_raw(texts)

def _embed_images(image_paths, batch_size=EMBED_BATCH):
    all_embs = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc="Embedding", leave=False):
        batch = image_paths[i : i + batch_size]
        imgs = []
        for p in batch:
            try: imgs.append(Image.open(p).convert("RGB"))
            except: imgs.append(Image.new("RGB", (IMG_SIZE, IMG_SIZE)))
        all_embs.append(_l2_norm(_encode_images_raw(imgs)))
        if DEVICE == "cuda": torch.cuda.empty_cache()
    return np.concatenate(all_embs, axis=0)

print("✓ Unified encode helpers ready.")


## Cell 6: Product Embeddings + FAISS Index

In [ ]:
def _ensure_valid_sets():
    """Rebuild valid sets from disk if needed (survives kernel restarts)."""
    global valid_product_set, valid_product_ids, valid_bundle_set, valid_bundle_ids
    prod_on_disk = {p.stem for p in PROD_DIR.glob("*.jpg")}
    bund_on_disk = {p.stem for p in BUND_DIR.glob("*.jpg")}
    if not valid_product_set and prod_on_disk:
        valid_product_set = prod_on_disk
        valid_product_ids = list(prod_on_disk)
        print(f"Rebuilt valid_product_set from disk: {len(valid_product_set):,}")
    if not valid_bundle_set and bund_on_disk:
        valid_bundle_set = bund_on_disk
        valid_bundle_ids = list(bund_on_disk)
        print(f"Rebuilt valid_bundle_set from disk: {len(valid_bundle_set):,}")
    print(f"valid_product_set: {len(valid_product_set):,}  valid_bundle_set: {len(valid_bundle_set):,}")

_ensure_valid_sets()

if EMB_FILE.exists() and IDS_FILE.exists():
    print("Loading cached product embeddings …")
    product_embeddings = np.load(EMB_FILE)
    with open(IDS_FILE) as f:
        indexed_product_ids: list[str] = json.load(f)
    print(f"  ✓ Loaded {product_embeddings.shape}")
else:
    print("Computing product embeddings (first run ~15–30 min on Colab T4) …")
    ordered_ids   = [pid for pid in valid_product_ids if pid in valid_product_set]
    ordered_paths = [PROD_DIR / f"{pid}.jpg" for pid in ordered_ids]
    product_embeddings = _embed_images(ordered_paths, batch_size=EMBED_BATCH)
    indexed_product_ids = ordered_ids
    np.save(EMB_FILE, product_embeddings)
    with open(IDS_FILE, "w") as f:
        json.dump(indexed_product_ids, f)
    print(f"  ✓ Saved embeddings: {product_embeddings.shape}")

idx_to_pid: dict[int, str] = {i: pid for i, pid in enumerate(indexed_product_ids)}
pid_to_idx: dict[str, int] = {pid: i for i, pid in idx_to_pid.items()}  # reverse needed for QE

DIM = product_embeddings.shape[1]
print(f"\nBuilding FAISS IndexFlatIP (dim={DIM}, n={len(indexed_product_ids):,}) …")
index_cpu = faiss.IndexFlatIP(DIM)
index_cpu.add(product_embeddings)

if DEVICE == "cuda":
    try:
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index_cpu)
        print("  Transferred index to GPU.")
    except Exception as e:
        print(f"  [WARN] GPU index failed ({e}), falling back to CPU.")
        index = index_cpu
else:
    index = index_cpu
    print("  Using CPU index.")

print(f"  FAISS ntotal = {index.ntotal:,}")
if index.ntotal < 1_000:
    raise RuntimeError(f"Only {index.ntotal} products indexed — re-check download step.")
elif index.ntotal < 20_000:
    print(f"  [WARN] Only {index.ntotal:,} products (expected ~27K). Results may be degraded.")


## Cell 7: Baseline Pipeline (whole-image retrieval)

In [ ]:
def predict_baseline(bundle_ids: list[str], k: int = TOP_K) -> dict[str, list[str]]:
    """Whole-image baseline: embed full bundle → top-k."""
    predictions: dict[str, list[str]] = {}
    for bid in tqdm(bundle_ids, desc="Baseline predict"):
        if bid not in valid_bundle_set:
            continue
        img = load_image(bid, BUND_DIR)
        if img is None:
            continue
        emb = _l2_norm(_encode_images_raw([img]))
        scores, indices = index.search(emb, k)
        top_pids = [idx_to_pid[int(i)] for i in indices[0] if int(i) in idx_to_pid]
        predictions[bid] = top_pids[:k]
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return predictions

print("✓ predict_baseline() defined.")


## Cell 8: SegFormer Segmentation

In [ ]:
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import torch.nn.functional as F

print("Loading SegFormer B2 clothes model …")
SEG_MODEL_ID = "mattmdjaga/segformer_b2_clothes"
seg_processor = SegformerImageProcessor.from_pretrained(SEG_MODEL_ID)
seg_model = SegformerForSemanticSegmentation.from_pretrained(SEG_MODEL_ID)
seg_model.to(DEVICE).eval()
print("✓ SegFormer loaded.")

ATR_LABELS = {
    0: "Background", 1: "Hat", 2: "Hair", 3: "Sunglasses", 4: "Upper-clothes",
    5: "Skirt", 6: "Pants", 7: "Dress", 8: "Belt", 9: "Left-shoe", 10: "Right-shoe",
    11: "Face", 12: "Left-leg", 13: "Right-leg", 14: "Left-arm", 15: "Right-arm",
    16: "Bag", 17: "Scarf",
}

SEGMENT_GROUPS: dict[str, list[int]] = {
    "upper_body": [4],
    "lower_body":  [5, 6],
    "dress":       [7],
    "shoes":       [9, 10],
    "bag":         [16],
    "hat":         [1],
    "scarf_belt":  [8, 17],
}

SEGMENT_TO_CATEGORIES: dict[str, set[str]] = {
    "upper_body": {"T-SHIRT", "SHIRT", "SWEATER", "WIND-JACKET", "TOPS AND OTHERS", "BLAZER", "SWEATSHIRT", "BABY T-SHIRT", "POLO SHIRT"},
    "lower_body": {"TROUSERS", "SKIRT", "BERMUDA", "BABY TROUSERS", "SHORTS", "LEGGINGS", "JEANS"},
    "dress":      {"DRESS", "OVERALL", "JUMPSUIT"},
    "shoes":      {"SHOE", "SHOES", "SNEAKER", "SNEAKERS", "BOOT", "BOOTS", "SANDAL", "SANDALS", "LOAFER"},
    "bag":        {"HAND BAG-RUCKSACK", "HANDBAG", "BAG", "BACKPACK", "PURSE", "CLUTCH"},
    "hat":        {"HAT", "CAP", "BEANIE"},
    "scarf_belt": {"SCARF", "BELT", "TIE"},
}

ATR_COLORS = plt.cm.get_cmap("tab20", 18)


@torch.no_grad()
def segment_image(pil_img: Image.Image) -> np.ndarray:
    inputs = seg_processor(images=pil_img, return_tensors="pt").to(DEVICE)
    logits = seg_model(**inputs).logits
    orig_h, orig_w = pil_img.height, pil_img.width
    upsampled = F.interpolate(logits, size=(orig_h, orig_w), mode="bilinear", align_corners=False)
    return upsampled.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.int32)


def extract_segment_crops(
    pil_img: Image.Image,
    seg_map: np.ndarray,
    min_frac: float = 0.05,
    padding: float = 0.05,
    extra_scales: list[float] | None = None,
) -> dict[str, list[Image.Image]]:
    """
    Returns dict: {segment_name: [crop, crop_scale2, crop_scale3, ...]}
    If MULTI_SCALE_CROPS is True and extra_scales is provided, returns crops at each scale.
    """
    W, H = pil_img.size
    total_pixels = H * W
    crops: dict[str, list[Image.Image]] = {}
    scales = extra_scales if (MULTI_SCALE_CROPS and extra_scales) else [1.0]

    for seg_name, label_ids in SEGMENT_GROUPS.items():
        mask = np.isin(seg_map, label_ids)
        if mask.sum() < min_frac * total_pixels:
            continue

        rows = np.where(mask.any(axis=1))[0]
        cols = np.where(mask.any(axis=0))[0]
        y_min, y_max = int(rows.min()), int(rows.max())
        x_min, x_max = int(cols.min()), int(cols.max())

        seg_crops = []
        for scale in scales:
            pad_y = int((y_max - y_min + 1) * padding * scale)
            pad_x = int((x_max - x_min + 1) * padding * scale)
            yn = max(0, y_min - pad_y)
            yx = min(H - 1, y_max + pad_y)
            xn = max(0, x_min - pad_x)
            xx = min(W - 1, x_max + pad_x)
            crop = pil_img.crop((xn, yn, xx + 1, yx + 1))
            if crop.width > 5 and crop.height > 5:
                seg_crops.append(crop)

        if seg_crops:
            crops[seg_name] = seg_crops

    return crops


def visualise_segmentation(pil_img, seg_map, bundle_id=""):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(pil_img); ax1.set_title("Original"); ax1.axis("off")
    cmap = ListedColormap([ATR_COLORS(i)[:3] for i in range(18)])
    ax2.imshow(seg_map, cmap=cmap, vmin=0, vmax=17); ax2.set_title("Segmentation"); ax2.axis("off")
    patches = [mpatches.Patch(color=ATR_COLORS(i)[:3], label=ATR_LABELS[i]) for i in range(18)]
    ax2.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=7)
    plt.tight_layout()
    plt.savefig(WORK_DIR / f"seg_{bundle_id[:10]}.png", bbox_inches="tight", dpi=80)
    plt.show()

print("✓ SegFormer helpers defined.")

# Demo on first test bundle
demo_bid = next((bid for bid in test_bundle_ids if bid in valid_bundle_set), None)
if demo_bid:
    demo_img = load_image(demo_bid, BUND_DIR)
    if demo_img:
        demo_seg = segment_image(demo_img)
        visualise_segmentation(demo_img, demo_seg, bundle_id=demo_bid)
        demo_crops = extract_segment_crops(demo_img, demo_seg)
        print(f"Segments found: {list(demo_crops.keys())}")


## Cell 9: Category Matching + Text Re-ranking Helpers

In [ ]:
def _category_match(desc: str, allowed: set) -> bool:
    """
    Substring category match (both directions) to handle plurals and compound names.
    e.g. 'SHOES' matches {'SHOE'}, 'HAND BAG-RUCKSACK' matches {'BAG'}.
    Empty descriptions pass through (unknown category → don't exclude).
    """
    if not desc.strip():
        return True  # unknown category: do not exclude
    d = desc.strip().upper()
    for cat in allowed:
        c = cat.upper()
        if d == c or d.startswith(c) or c.startswith(d) or c in d or d in c:
            return True
    return False


def text_rerank(
    candidate_pids: list[str],
    crop_embs: np.ndarray,
    alpha: float = TEXT_RERANK_ALPHA,
) -> list[str]:
    """Linearly combine image rank score with text–image cross-similarity."""
    if not candidate_pids:
        return candidate_pids
    descriptions = [product_desc.get(pid, "") for pid in candidate_pids]
    non_empty = [(i, d) for i, d in enumerate(descriptions) if d.strip()]
    if not non_empty:
        return candidate_pids

    indices_ne, descs_ne = zip(*non_empty)
    text_embs = _l2_norm(_encode_texts_raw(list(descs_ne)))  # (n_text, 512)
    sim = crop_embs @ text_embs.T                            # (n_crops, n_text)
    text_scores_ne = sim.max(axis=0)                         # (n_text,)

    img_rank_score = np.array([1.0 - i / len(candidate_pids) for i in range(len(candidate_pids))])
    text_score_full = np.zeros(len(candidate_pids))
    for list_idx, orig_idx in enumerate(indices_ne):
        text_score_full[orig_idx] = float(text_scores_ne[list_idx])

    combined = (1 - alpha) * img_rank_score + alpha * text_score_full
    return [candidate_pids[j] for j in np.argsort(combined)[::-1]]


def _aggregate_scores(
    pid_rank_lists: list[list[str]],
    pid_score_lists: list[np.ndarray],
    use_rrf: bool = False,
    rrf_k: int = RRF_K,
    normalize: bool = False,
) -> dict[str, float]:
    """
    Aggregate per-segment score/rank lists into a single scores dict.

    Modes:
      use_rrf=False, normalize=False  → raw cosine sum  (original)
      use_rrf=False, normalize=True   → min-max normalised cosine sum
      use_rrf=True                    → Reciprocal Rank Fusion (ignores scores)
    """
    scores_map: dict[str, float] = {}

    for pids, scores in zip(pid_rank_lists, pid_score_lists):
        if use_rrf:
            for rank, pid in enumerate(pids):
                scores_map[pid] = scores_map.get(pid, 0.0) + 1.0 / (rrf_k + rank + 1)
        else:
            if normalize and len(scores) > 1:
                lo, hi = scores.min(), scores.max()
                scores = (scores - lo) / (hi - lo + 1e-8) if hi > lo else scores
            for pid, sc in zip(pids, scores):
                scores_map[pid] = scores_map.get(pid, 0.0) + float(sc)

    return scores_map


print("✓ _category_match(), text_rerank(), _aggregate_scores() defined.")


## Cell 10: Standard Segmented Pipeline (backport of Kaggle fixes)

In [ ]:
def predict_segmented(
    bundle_ids: list[str],
    k: int = TOP_K,
) -> dict[str, list[str]]:
    """
    Standard per-segment retrieval (Kaggle-equivalent):
    SegFormer → crops → FAISS per segment → category filter + fallback
    → sum aggregation → text re-ranking → top-k.
    """
    predictions: dict[str, list[str]] = {}

    for bid in tqdm(bundle_ids, desc="Segmented predict"):
        if bid not in valid_bundle_set:
            continue
        img = load_image(bid, BUND_DIR)
        if img is None:
            continue

        try:
            seg_map = segment_image(img)
            crops_dict = extract_segment_crops(img, seg_map)
        except Exception as e:
            print(f"  [WARN] Segmentation failed for {bid}: {e}")
            crops_dict = {}

        if crops_dict:
            # Use first (base-padded) crop only in standard mode
            crop_list = [v[0] for v in crops_dict.values()]
            seg_names = list(crops_dict.keys())
        else:
            crop_list = [img]
            seg_names = ["whole"]

        embs_norm = _l2_norm(_encode_images_raw(crop_list))

        # FAISS query per segment
        scores_arr, indices_arr = index.search(embs_norm, K_PER_SEGMENT)

        pid_rank_lists = []
        pid_score_lists = []

        for seg_name, seg_scores, seg_indices in zip(seg_names, scores_arr, indices_arr):
            allowed = SEGMENT_TO_CATEGORIES.get(seg_name, set())
            filtered, unfiltered = [], []
            for sc, idx_val in zip(seg_scores, seg_indices):
                pid = idx_to_pid.get(int(idx_val))
                if pid is None: continue
                unfiltered.append((pid, float(sc)))
                if not allowed or _category_match(product_desc.get(pid, ""), allowed):
                    filtered.append((pid, float(sc)))

            to_add = filtered if filtered else [(p, s * 0.50) for p, s in unfiltered[:K_PER_SEGMENT // 4]]
            pids_seg  = [p for p, _ in to_add]
            scores_seg = np.array([s for _, s in to_add])
            pid_rank_lists.append(pids_seg)
            pid_score_lists.append(scores_seg)

        scores_map = _aggregate_scores(pid_rank_lists, pid_score_lists, use_rrf=False, normalize=False)

        ranked = sorted(scores_map.items(), key=lambda x: x[1], reverse=True)
        top_pids = [pid for pid, _ in ranked[:TEXT_RERANK_TOP_N]]

        if ENABLE_TEXT_RERANK and top_pids:
            top_pids = text_rerank(top_pids, embs_norm, alpha=TEXT_RERANK_ALPHA)

        predictions[bid] = top_pids[:k]

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    return predictions


print("✓ predict_segmented() defined.")


## Cell 11: Experimental Pipeline — `predict_segmented_v2`

Combines all new techniques:
- **Multi-scale crops** (1.0×, 1.3×, 1.7× padding) — averaged embedding per segment
- **Reciprocal Rank Fusion** instead of raw score sum
- **Score normalization** before aggregation
- **Whole-image ensemble** — blend full-bundle cosine score (weight=`ENSEMBLE_WHOLE_WEIGHT`)
- **Query Expansion** — top-`QE_TOP_N` products re-embedded for a second FAISS pass

All experiment flags are in Cell 2 config.


In [ ]:
def predict_segmented_v2(
    bundle_ids: list[str],
    k: int = TOP_K,
) -> dict[str, list[str]]:
    """
    Experimental per-segment retrieval with all Colab improvements:
      1. Multi-scale crops per segment (averaged embeddings)
      2. Category filter + graceful fallback
      3. RRF or normalised-sum aggregation
      4. Whole-image ensemble blending
      5. Query expansion via top retrieved product images
      6. Text re-ranking on final candidates
    """
    predictions: dict[str, list[str]] = {}

    for bid in tqdm(bundle_ids, desc="Segmented v2"):
        if bid not in valid_bundle_set:
            continue
        img = load_image(bid, BUND_DIR)
        if img is None:
            continue

        # ── Segmentation ──────────────────────────────────────────────────────
        try:
            seg_map = segment_image(img)
            crops_dict = extract_segment_crops(
                img, seg_map,
                extra_scales=SCALE_FACTORS if MULTI_SCALE_CROPS else [1.0],
            )
        except Exception as e:
            print(f"  [WARN] Segmentation failed for {bid}: {e}")
            crops_dict = {}

        if not crops_dict:
            crops_dict = {"whole": [img]}

        # ── Embedding: average over scales per segment ────────────────────────
        seg_names: list[str] = []
        embs_list: list[np.ndarray] = []   # one (1,512) per segment

        for seg_name, scale_crops in crops_dict.items():
            raw = _encode_images_raw(scale_crops)          # (n_scales, 512)
            avg_emb = _l2_norm(raw.mean(axis=0, keepdims=True))  # (1, 512)
            seg_names.append(seg_name)
            embs_list.append(avg_emb)

        embs_norm = np.concatenate(embs_list, axis=0)  # (n_segs, 512)

        # ── FAISS per segment ─────────────────────────────────────────────────
        scores_arr, indices_arr = index.search(embs_norm, K_PER_SEGMENT)

        pid_rank_lists: list[list[str]] = []
        pid_score_lists: list[np.ndarray] = []

        for seg_name, seg_scores, seg_indices in zip(seg_names, scores_arr, indices_arr):
            allowed = SEGMENT_TO_CATEGORIES.get(seg_name, set())
            filtered, unfiltered = [], []
            for sc, idx_val in zip(seg_scores, seg_indices):
                pid = idx_to_pid.get(int(idx_val))
                if pid is None: continue
                unfiltered.append((pid, float(sc)))
                if not allowed or _category_match(product_desc.get(pid, ""), allowed):
                    filtered.append((pid, float(sc)))

            to_add = filtered if filtered else [(p, s * 0.50) for p, s in unfiltered[:K_PER_SEGMENT // 4]]
            pids_seg   = [p for p, _ in to_add]
            scores_seg = np.array([s for _, s in to_add])
            pid_rank_lists.append(pids_seg)
            pid_score_lists.append(scores_seg)

        # ── Aggregation ───────────────────────────────────────────────────────
        scores_map = _aggregate_scores(
            pid_rank_lists,
            pid_score_lists,
            use_rrf=ENABLE_RRF,
            normalize=NORMALIZE_SEGMENT_SCORES,
        )

        # ── Whole-image ensemble ──────────────────────────────────────────────
        if ENSEMBLE_WHOLE_IMAGE:
            whole_emb = _l2_norm(_encode_images_raw([img]))  # (1, 512)
            wi_scores, wi_indices = index.search(whole_emb, K_PER_SEGMENT)
            wi_max = float(wi_scores.max()) if wi_scores.max() > 0 else 1.0
            for sc, idx_val in zip(wi_scores[0], wi_indices[0]):
                pid = idx_to_pid.get(int(idx_val))
                if pid is None: continue
                scores_map[pid] = scores_map.get(pid, 0.0) + ENSEMBLE_WHOLE_WEIGHT * float(sc) / wi_max

        # ── Query Expansion ───────────────────────────────────────────────────
        if ENABLE_QUERY_EXPANSION:
            # Sort current candidates, take top-QE_TOP_N products, embed their images
            top_for_qe = sorted(scores_map.items(), key=lambda x: x[1], reverse=True)[:QE_TOP_N]
            qe_imgs = []
            for qe_pid, _ in top_for_qe:
                qe_img = load_image(qe_pid, PROD_DIR)
                if qe_img is not None:
                    qe_imgs.append(qe_img)

            if qe_imgs:
                qe_embs = _l2_norm(_encode_images_raw(qe_imgs))   # (n_qe, 512)
                qe_query = _l2_norm(qe_embs.mean(axis=0, keepdims=True))  # (1, 512)

                # Blend: alpha * original crop embs + (1-alpha) * expanded query
                blended_query = _l2_norm(
                    QE_ALPHA * embs_norm.mean(axis=0, keepdims=True) +
                    (1 - QE_ALPHA) * qe_query
                )  # (1, 512)

                qe_scores, qe_indices = index.search(blended_query, K_PER_SEGMENT)
                qe_max = float(qe_scores.max()) if qe_scores.max() > 0 else 1.0

                for sc, idx_val in zip(qe_scores[0], qe_indices[0]):
                    pid = idx_to_pid.get(int(idx_val))
                    if pid is None: continue
                    # Add half-weight query expansion bonus
                    scores_map[pid] = scores_map.get(pid, 0.0) + 0.5 * float(sc) / qe_max

        # ── Text re-ranking ───────────────────────────────────────────────────
        ranked = sorted(scores_map.items(), key=lambda x: x[1], reverse=True)
        top_pids = [pid for pid, _ in ranked[:TEXT_RERANK_TOP_N]]

        if ENABLE_TEXT_RERANK and top_pids:
            top_pids = text_rerank(top_pids, embs_norm, alpha=TEXT_RERANK_ALPHA)

        predictions[bid] = top_pids[:k]

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    return predictions


print("✓ predict_segmented_v2() defined.")


## Cell 12: Validation (Recall@15)

In [ ]:
def recall_at_k(predictions: dict[str, list[str]], ground_truth: dict[str, set], k: int = TOP_K) -> float:
    """Macro-averaged Recall@k over all bundles with GT labels."""
    total_recall, n = 0.0, 0
    for bid, gt_set in ground_truth.items():
        if not gt_set: continue
        preds = predictions.get(bid, [])[:k]
        total_recall += len(set(preds) & gt_set) / len(gt_set)
        n += 1
    return total_recall / n if n > 0 else 0.0


_ensure_valid_sets()
train_bundle_ids_with_gt = list(train_gt.keys())

print("Running baseline on train bundles …")
baseline_train_preds = predict_baseline(train_bundle_ids_with_gt)

print("\nRunning standard segmented on train bundles …")
seg_train_preds = predict_segmented(train_bundle_ids_with_gt)

print("\nRunning experimental v2 on train bundles …")
seg_v2_train_preds = predict_segmented_v2(train_bundle_ids_with_gt)

recall_base = recall_at_k(baseline_train_preds, train_gt)
recall_seg  = recall_at_k(seg_train_preds, train_gt)
recall_v2   = recall_at_k(seg_v2_train_preds, train_gt)

print(f"\n{'='*50}")
print(f"Recall@{TOP_K}  Baseline    : {recall_base:.4f}")
print(f"Recall@{TOP_K}  Segmented   : {recall_seg:.4f}")
print(f"Recall@{TOP_K}  Seg v2 (exp): {recall_v2:.4f}")
print(f"Delta seg→v2 : {recall_v2 - recall_seg:+.4f}")
print(f"{'='*50}")

# ── Per-complexity breakdown ──────────────────────────────────────────────────
complexity_bins = {
    "easy (1-2)":   [b for b, g in train_gt.items() if 1 <= len(g) <= 2],
    "medium (3-5)": [b for b, g in train_gt.items() if 3 <= len(g) <= 5],
    "hard (6+)":    [b for b, g in train_gt.items() if len(g) >= 6],
}

print("\nPer-complexity Recall@15:")
labels, base_vals, seg_vals, v2_vals = [], [], [], []
for label, bids in complexity_bins.items():
    sub_gt = {b: train_gt[b] for b in bids if b in train_gt}
    r_base = recall_at_k(baseline_train_preds, sub_gt)
    r_seg  = recall_at_k(seg_train_preds, sub_gt)
    r_v2   = recall_at_k(seg_v2_train_preds, sub_gt)
    print(f"  {label:16s}  Base={r_base:.3f}  Seg={r_seg:.3f}  V2={r_v2:.3f}  n={len(bids)}")
    labels.append(label); base_vals.append(r_base); seg_vals.append(r_seg); v2_vals.append(r_v2)

x = np.arange(len(labels))
w = 0.25
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w, base_vals, w, label="Baseline",    color="steelblue")
ax.bar(x,     seg_vals,  w, label="Segmented",   color="coral")
ax.bar(x + w, v2_vals,   w, label="Seg v2 (exp)", color="mediumseagreen")
ax.set_ylabel("Recall@15"); ax.set_title("Recall@15 by Bundle Complexity")
ax.set_xticks(x); ax.set_xticklabels(labels); ax.legend()
plt.tight_layout()
plt.savefig(WORK_DIR / "recall_comparison_v2.png", bbox_inches="tight", dpi=100)
plt.show()


## Cell 13: Generate Submission

In [ ]:
# Frequency-based fallback (most common products in training set)
product_freq = Counter(
    pid for gt_set in train_gt.values() for pid in gt_set
    if pid in valid_product_set
)
fallback_pids = [pid for pid, _ in product_freq.most_common(TOP_K)]
print(f"Fallback products: {fallback_pids[:5]} …")

# Generate test predictions for all methods
print("\nRunning segmented v2 on test bundles …")
segmented_preds    = predict_segmented(test_bundle_ids, k=TOP_K)
segmented_v2_preds = predict_segmented_v2(test_bundle_ids, k=TOP_K)
baseline_preds     = predict_baseline(test_bundle_ids, k=TOP_K)

# Auto-select best method based on train recall
method_scores = {
    "baseline":    recall_base,
    "segmented":   recall_seg,
    "segmented_v2": recall_v2,
}
best_method = max(method_scores, key=method_scores.get)
best_preds  = {"baseline": baseline_preds, "segmented": segmented_preds, "segmented_v2": segmented_v2_preds}[best_method]
print(f"\nSelected method: {best_method} (train Recall@{TOP_K} = {method_scores[best_method]:.4f})")
print(f"All scores: { {k: f'{v:.4f}' for k, v in method_scores.items()} }")

# Build submission rows
rows: list[dict] = []
missing_count = 0
for bid in test_bundle_ids:
    preds = list(best_preds.get(bid, []))
    if not preds:
        preds = fallback_pids[:]
        missing_count += 1
    # Pad to TOP_K
    for fp in fallback_pids:
        if fp not in preds:
            preds.append(fp)
        if len(preds) >= TOP_K:
            break
    for pid in preds[:TOP_K]:
        rows.append({"bundle_asset_id": bid, "product_asset_id": pid})

submission_df = pd.DataFrame(rows)
submission_df.to_csv(SUBM_FILE, index=False)
print(f"\n✓ Submission saved: {SUBM_FILE}")
print(f"  Rows: {len(submission_df):,}   Missing-bundle fallbacks: {missing_count}")

# ── Sanity checks ─────────────────────────────────────────────────────────────
covered = set(submission_df["bundle_asset_id"].unique())
missing = set(test_bundle_ids) - covered
assert len(missing) == 0, f"Missing bundles: {missing}"
print(f"  [OK] All {len(test_bundle_ids)} test bundles present.")

max_preds = submission_df.groupby("bundle_asset_id")["product_asset_id"].count().max()
assert max_preds <= TOP_K, f"Bundle has {max_preds} predictions (max={TOP_K})"
print(f"  [OK] Max predictions per bundle: {max_preds}")

all_catalog_pids = set(products_df["product_asset_id"])
invalid_pids = set(submission_df["product_asset_id"]) - all_catalog_pids
if invalid_pids:
    print(f"  [WARN] {len(invalid_pids)} invalid product IDs — removing + refilling.")
    submission_df = submission_df[~submission_df["product_asset_id"].isin(invalid_pids)]
    existing_by_bundle = submission_df.groupby("bundle_asset_id")["product_asset_id"].apply(list).to_dict()
    refill_rows = []
    for bid in test_bundle_ids:
        curr = existing_by_bundle.get(bid, [])
        for fp in fallback_pids:
            if len(curr) >= TOP_K: break
            if fp not in curr:
                curr.append(fp)
                refill_rows.append({"bundle_asset_id": bid, "product_asset_id": fp})
    if refill_rows:
        submission_df = pd.concat([submission_df, pd.DataFrame(refill_rows)], ignore_index=True)
    submission_df.to_csv(SUBM_FILE, index=False)
    print(f"  Refilled {len(refill_rows)} predictions.")
else:
    print(f"  [OK] All {len(set(submission_df['product_asset_id'])):,} product IDs valid.")

print("\nPreview:")
print(submission_df.head(10).to_string(index=False))


## Cell 14: Summary

In [ ]:
print("\n" + "=" * 60)
print("HACKUDC 2026 — COLAB EXPERIMENT SUMMARY")
print("=" * 60)
print(f"Products indexed          : {index.ntotal:,}")
print(f"Embedding dimension       : {DIM}")
print(f"FAISS backend             : {'GPU' if DEVICE == 'cuda' else 'CPU'}")
print(f"Segmentation model        : SegFormer B2 (ATR 17 classes)")
print(f"Retrieval model           : {'Marqo-FashionSigLIP' if USE_MARQO else 'FashionCLIP'} (512-dim)")
print(f"")
print(f"Recall@{TOP_K}  Baseline      : {recall_base:.4f}")
print(f"Recall@{TOP_K}  Segmented     : {recall_seg:.4f}")
print(f"Recall@{TOP_K}  Seg v2 (exp)  : {recall_v2:.4f}")
print(f"")
print(f"Experiments active:")
print(f"  Multi-scale crops       : {MULTI_SCALE_CROPS}  (scales={SCALE_FACTORS})")
print(f"  RRF aggregation         : {ENABLE_RRF}        (k={RRF_K})")
print(f"  Score normalisation     : {NORMALIZE_SEGMENT_SCORES}")
print(f"  Whole-image ensemble    : {ENSEMBLE_WHOLE_IMAGE}  (weight={ENSEMBLE_WHOLE_WEIGHT})")
print(f"  Query expansion         : {ENABLE_QUERY_EXPANSION}  (top={QE_TOP_N}, alpha={QE_ALPHA})")
print(f"  Text re-ranking         : {ENABLE_TEXT_RERANK}  (alpha={TEXT_RERANK_ALPHA})")
print(f"")
print(f"Best method (train)       : {best_method}")
print(f"Submission file           : {SUBM_FILE}")
print(f"Submission rows           : {len(submission_df):,}")
print("=" * 60)
